# Ten-column boundary representation

This study compares predictor representations and the public specification search on the original ten-column boundary task in [#608](https://github.com/fcela/mboostrs/issues/608). Each of 50 independent confirmation draws contains 2,000 observations with ten independent Uniform(−1,1) predictors and response $Y=3\operatorname{sign}(x_0x_1)+x_2+\epsilon$, where $\epsilon\sim N(0,0.5^2)$. The original 70% fitting/30% evaluation split uses `random_state=0`. Normalized root mean squared error (NRMSE) divides evaluation RMSE by the evaluation response population standard deviation.

The [protocol](all_column_boundary/PROTOCOL.md) fixes all eight methods before measurement. CatBoost with 200 trees is the primary reference. The target is at most 1% paired NRMSE excess; a point estimate and a one-sided 95% upper bound are reported separately. Ten development draws and 50 fresh confirmation draws are retained separately. No test result chooses a representation or changes an estimator.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
from docs.book.cases.bench_tables import table

directory = root / "docs/book/benchmarks/all_column_boundary"
summary = json.loads((directory / "confirmation_summary.json").read_text())
facts = json.loads((directory / "confirmation_validation.json").read_text())
labels = {"original_hybrid": "Original hybrid", "additive": "Additive smooths",
          "supplied_pair": "Supplied pair", "supplied_pair_all_mains": "Pair and all main effects",
          "supplied_tuned": "Supplied tuned pair", "discovery": "Data-only discovery",
          "catboost": "CatBoost 200", "lightgbm": "LightGBM 200"}

In [ ]:
headline = pl.DataFrame([{
    "Method": labels[r["method"]], "NRMSE": r["nrmse"], "NRMSE SE": r["nrmse_se"],
    "Mean MSE": r["true_mean_mse"], "Seconds": r["elapsed"], "Failures": r["failures"]}
    for r in summary])
table(headline, title="Every method on the original ten-column task",
      subtitle="50 fresh draws; smaller NRMSE, latent-mean MSE and elapsed seconds are better",
      stub="Method", stubhead="Method", better={"NRMSE": "min", "Mean MSE": "min", "Seconds": "min"},
      spanners={"Prediction error": ["NRMSE", "NRMSE SE", "Mean MSE"]},
      notes=["NRMSE SE is the Monte Carlo standard error across independent draws. Mean MSE compares the "
             "prediction with the generating mean in squared response units. Seconds measure the complete "
             "workflow on shared hardware, including tuning for tuned/discovery methods; they are not controlled "
             "implementation speed ratios. Bold marks descriptive minima, not a post hoc method selection.",
             "Failures count incomplete draws; a method with a failure would have no successful-only average. "
             "Source: confirmation_summary.json, which summarize_all_column_boundary.py computes from the 400 "
             "per-draw records written by study_all_column_boundary.py."])

In [ ]:
by_method = {r["method"]: r for r in summary}
comparison = pl.DataFrame([{
    "Method": labels[r["method"]], "Excess %": r["catboost_excess"],
    "Paired SE": r["catboost_se"], "Upper 95%": r["catboost_upper95"],
    "Adjusted upper": r["catboost_upper_adjusted95"], "Target": r["upper_target"]}
    for r in summary])
table(comparison, title="Paired NRMSE excess over prespecified CatBoost",
      subtitle="Percentage points; smaller is better; target upper bound ≤ 1%",
      stub="Method", stubhead="Method", better={"Excess %": "min"},
      notes=["Each draw contributes 100 × (method NRMSE / CatBoost NRMSE − 1). Bounds use Student-t over 50 "
             "independent paired draws. Adjusted upper uses the two-comparator Bonferroni critical value; "
             "the separate LightGBM comparison is retained in the summary. Target is the unadjusted upper criterion. "
             "The prespecified target procedures are supplied tuned pair and data-only discovery."])

The supplied tuned pair has a mean paired NRMSE excess of {eval}`f"{by_method['supplied_tuned']['catboost_excess']:.2f}%"`, with upper bound {eval}`f"{by_method['supplied_tuned']['catboost_upper95']:.2f}%"`. Data-only discovery has excess {eval}`f"{by_method['discovery']['catboost_excess']:.2f}%"`, with upper bound {eval}`f"{by_method['discovery']['catboost_upper95']:.2f}%"`. Both prespecified workflows meet the target on this task. The original hybrid remains worse, at {eval}`f"{by_method['original_hybrid']['catboost_excess']:.2f}%"` excess. The study changes no learner engine and does not show that the original formula improved.

## Predictor representations

The original hybrid combines all ten default `bbs` smooths with `btree(x0, ..., x9, maxdepth=6)` for 500 updates at learning rate 0.1. The additive control removes the tree. Two supplied controls restrict the tree to `x0,x1`, depth 3 and `selection=sse`, with either `bols(x2)` or all ten smooths. The supplied tuned target uses the pair plus `bols(x2)`, five-fold stopping selection, starting budget 100 and cap 500 at rate 0.1. These supplied formulas use knowledge of the generating mechanism.

Data-only discovery receives all ten predictors, no family or formula, seed 0 and its default cap 400. It must identify the useful interaction itself. CatBoost uses 200 trees, depth 6 and learning rate 0.05; LightGBM uses 200 trees, 31 leaves, unlimited depth and rate 0.05. Both competitors use seed 0 and one thread. Their settings are fixed rather than tuned to these test draws.

A `btree` term already fits successive gradient trees during boosting. The earlier gap cannot be attributed to the absence of an ensemble. This experiment supports the pair representation and the public selection workflow for this generating process; it does not establish universal tree capacity.

## Public fitting workflow

The example repeats one frozen confirmation input to illustrate fitting, stopping selection and prediction. It adds no independent evidence to the comparison. The five inner folds choose the number of updates using fitting rows only; the evaluation response is used after prediction.

In [ ]:
import mboostrs as mb
from benches.study_all_column_boundary import draw

frame, true_mean, fitting_rows, testing_rows = draw(6088100)
fitting = frame[fitting_rows]
evaluation = frame[testing_rows]

selected = mb.tune_mstop(
    "y ~ btree(x0, x1, maxdepth=3, selection=sse) + bols(x2)", fitting,
    family=mb.Gaussian(), nu=0.1, folds=5, start=100, max_mstop=500, seed=0)
prediction = selected.model.predict(evaluation.drop("y"), type="response")
example_mse = float(np.mean((prediction - evaluation["y"].to_numpy()) ** 2))
example = pl.DataFrame({"Observed": evaluation["y"].head(5), "Predicted": prediction[:5]})
(GT(example).tab_header(title="Supplied tuned predictions", subtitle="Response units; first five evaluation observations")
 .fmt_number(columns=["Observed", "Predicted"], decimals=3)
 .tab_source_note(f"Selected updates: {selected.mstop}; evaluation MSE: {example_mse:.4f}. "
                  "This is a demonstration on a previously recorded input."))

The corresponding data-only call is `mb.discover(fitting, "y", seed=0)`. A returned distributional model reports a dictionary of predictions, from which the mean is `prediction["mu"]`; a standard model returns the mean array directly. The following table shows the formula and family the search selected on the confirmation input used in the preceding example. It reads the `first_draw_selection` entry that `summarize_all_column_boundary.py` copies into `confirmation_summary.json` from that draw's stored discovery record.

In [ ]:
# The summarizer copies the data-only choices on the first confirmation draw into the discovery row.
choices = by_method["discovery"]["first_draw_selection"]
choice_frame = pl.DataFrame({"Field": ["Seed", "Family", "Mean formula", "Spread formula", "Stopping budget"],
    "Value": [str(choices["seed"]), str(choices["family"]), str(choices["formula"]),
              str(choices["spread_formula"]), str(choices["mstop"])]})
(GT(choice_frame).tab_header(title="First frozen data-only selection",
                           subtitle="The search sees all ten observed predictors")
 .tab_source_note("Source: confirmation_summary.json. The per-draw record in confirmation_a.jsonl.xz also holds the full "
                  "search log, provenance, predictions, warnings and row identities."))

## Evidence and validation

Development seeds 6087100–6087109 precede confirmation seeds 6088100–6088149. The two confirmation archives contain disjoint batches of 25 draws. Every method within a draw shares the fitting/evaluation rows and the same generated data. The validator checks compressed and uncompressed SHA-256 digests, exactly eight records per draw, 1,400/600 disjoint row identities, and recomputes MSE, latent-mean MSE and NRMSE from stored predictions to absolute tolerance $10^{-12}$.

All failures, warnings, model choices, elapsed costs and source/native identities are retained. The 80 development records and all 400 confirmation records completed. Gaussian scale attribution remains a separate question in [#629](https://github.com/fcela/mboostrs/issues/629): accurate mean prediction does not validate a selected noise-scale explanation.

In [ ]:
development = json.loads((directory / "development_summary.json").read_text())
phases = pl.DataFrame([{"Phase": phase, "Method": labels[r["method"]], "NRMSE": r["nrmse"],
                       "SE": r["nrmse_se"], "Failures": r["failures"]}
    for phase, records in (("Development (10)", development), ("Confirmation (50)", summary)) for r in records])
table(phases, title="Development and confirmation reported separately",
      subtitle="NRMSE and Monte Carlo SE; lower NRMSE is better", stub="Method", stubhead="Method", group="Phase",
      better={"NRMSE": "min"}, notes=["Source: development_summary.json and confirmation_summary.json. "
      "No confirmation result changes the predefined methods or budgets; failures are not excluded from aggregates."])

## Reproduction

Use the repository-local Python environment and a release extension matching the source revision recorded in each metadata file. The supplied runner refuses to overwrite evidence. Choose a new output directory or preserve the existing files before a reproduction. From the repository root:

```bash
PYTHONPATH=python:. .venv/bin/python benches/study_all_column_boundary.py \
  --phase development --output /tmp/boundary-development.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/study_all_column_boundary.py \
  --phase confirmation --output /tmp/boundary-confirmation.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/summarize_all_column_boundary.py --phase confirmation
```

The final command validates the per-draw archives, which must first be downloaded as described below. Their metadata records the runner hash, loaded Python source hashes, native build identity, package versions, thread settings and protocol hash. Reproducing on newer code is a new run and does not replace the frozen result. The downloadable notebook bundle includes the protocol, metadata, summary and validation files listed below. The per-draw archives are assets of the `benchmark-raw-archives-1` GitHub release; `docs/book/benchmarks/raw_archives.json` lists each file with its size, SHA-256 and the command that regenerates it. They are not bundled.

In [ ]:
# Per-draw archives are release assets listed in raw_archives.json; the bundle holds the files this page reads or cites.
artifacts = sorted(p.name for p in directory.iterdir()
                   if p.is_file() and not p.name.endswith((".jsonl.xz", ".jsonl.gz")))
(GT(pl.DataFrame({"Artifact": artifacts}))
 .tab_header(title="Bundled evidence", subtitle="Protocol, run metadata, summaries and validation")
 .tab_source_note("Each metadata file records the SHA-256 digests of one per-draw archive; the summarizer checks "
                  "them before recomputing every score from stored predictions."))